## The Critic

The critic is simply another neural network that predicts state values.

### Input

$$
s_t
$$

### Output

$$
V_\phi(s_t)
$$

This means: "From this state, how much total future reward do I expect?"

### Example

```text
state S3
  ↓
critic
  ↓
V(S3) = 4.7
```

### Roles

- The critic estimates expected returns.
- The actor chooses actions based on the advantage (which uses the critic's estimate).

The critic is not choosing actions directly.

## 3. Advantage

Our Monte-Carlo estimate of advantage is:

$$
\hat{A}_t = G_t - V_\phi(s_t)
$$

### Example: Good outcome

Suppose:

- $G_t = 8$ (actual return)
- $V(s_t) = 5$ (critic's estimate)

Then:

$$
A_t = 8 - 5 = 3
$$

This means: the outcome was 3 better than expected.

### Example: Bad outcome

Suppose:

- $G_t = 2$ (actual return)
- $V(s_t) = 5$ (critic's estimate)

Then:

$$
A_t = 2 - 5 = -3
$$

This means: the outcome was 3 worse than expected.

## 4. Actor Loss

Instead of using the raw return:

$$
L_{\text{actor}} = -G_t \log \pi_\theta(a_t \mid s_t)
$$

we use the advantage:

$$
L_{\text{actor}} = -\hat{A}_t \log \pi_\theta(a_t \mid s_t)
$$

### The logic

- If $A > 0$: increase the probability of that action.
- If $A < 0$: decrease the probability of that action.

This is much cleaner than blindly using raw returns.

## 5. Critic Loss

The critic itself needs training.

### Goal

We want:

$$
V_\phi(s_t) \approx G_t
$$

### Simple loss function

$$
L_{\text{critic}} = (V_\phi(s_t) - G_t)^2
$$

This is ordinary squared regression.

### Example

- Actual return: $G_t = 10$
- Critic's prediction: $V_\phi(s_t) = 6$
- Loss: $L = (6 - 10)^2 = 16$

The critic learns to predict values closer to the actual return.

## The Interplay Between Actor and Critic

Notice something interesting about the relationship between actor and critic.

### How they interact

The actor uses:

$$
A = G - V(s)
$$

But the critic is itself learning:

$$
V(s) \to G
$$

### Early training (critic is terrible)

Suppose:

- $G = 1$ (actual return)
- $V = -7$ (terrible critic estimate)

Then:

$$
A = 1 - (-7) = 8
$$

The actor receives a huge signal.

### Later training (critic improves)

Suppose:

- $G = 1$ (actual return)
- $V = 0.98$ (good critic estimate)

Then:

$$
A = 1 - 0.98 = 0.02
$$

The actor receives a much smaller signal.

### Why this is good

As the critic improves at predicting the expected outcome, the actor receives a more precise "better/worse than expected" signal.

## The Deeper Insight

Suppose every action from a state tends to produce a high reward simply because the state is easy.

### With raw REINFORCE

Raw returns:

- Action A → $+100$
- Action B → $+95$
- Action C → $+105$

All three actions receive massive positive reinforcement, even though they are not meaningfully different.

### With Actor-Critic

Suppose the critic learns:

$$
V(s) \approx 100
$$

Then the advantages are:

- Action A → $100 - 100 = 0$ (neutral)
- Action B → $95 - 100 = -5$ (bad)
- Action C → $110 - 100 = +10$ (good)

Now the policy learns the correct relative ranking:

- C is better than expected
- A is normal
- B is worse

That is the signal we actually want.

## Where We Are Now

### The Policy Optimization Progression

We have built up the theory step by step:

```text
REINFORCE
  ↓ (raw return causes high variance)
Baseline
  ↓ (G - V(s) centers advantage)
Advantage
  ↓ (advantage replaces raw return)
Actor-Critic
  ↓ (policy updates still potentially too aggressive)
PPO
```

### From Theory to LLM Applications

After PPO, the progression extends to practical LLM applications:

```text
PPO
  ↓
LLM token-level policy
  ↓
reward model / verifier
  ↓
reasoning RL
  ↓
GRPO
  ↓
tool-use / multi-step agents
```

This represents the full journey from basic policy gradient methods to sophisticated multi-turn reasoning agents.